# Stuttering Detection: Advanced Visual Research & Diagnostics
**Course**: CS204T (Artificial Intelligence)  
**Team**: 18  
**Objective**: Proving feature separability and visualizing model decision logic through PCA, t-SNE, and Boundary Maps.

---

## Step 1: Environment & Full Scale Data Loading
We load the **Full Extracted Dataset** (28,000 clips) to ensure our visualizations cover the entire acoustic diversity of the podcasts.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA

from src.data import DataManager
from src.models import (
    LogisticModel, LinearSVMModel, 
    KernelSVMModel, ShallowNeuralNetwork
)

# Paths
FEATURE_DIR = "data/features"
fluent_dir = os.path.join(FEATURE_DIR, "fluent")
disfluent_dir = os.path.join(FEATURE_DIR, "disfluent")

# 1. Load distributed data matrices
manager = DataManager(None, None)
X, y = manager.load_from_folders(fluent_dir, disfluent_dir)

# --- NEW: Global Shuffle & Subset to 10k for Speed ---
import numpy as np
np.random.seed(42)
indices = np.random.permutation(len(y))
X = X[indices][:10000]
y = y[indices][:10000]
manager.analyze_distribution()

## Step 2: PCA - Information Audit & Dimensionality Reduction
WavLM generates 768 features. We use PCA to determine how many of these are truly unique signals vs redundant noise. We target **95% Explained Variance**.

In [ ]:
# Standardize features before PCA
X_scaled = manager.preprocess(X, method="standard", fit=True)

# Audit Variance
full_pca = PCA().fit(X_scaled)
cum_variance = np.cumsum(full_pca.explained_variance_ratio_)

plt.figure(figsize=(10, 6))
plt.plot(cum_variance, lw=2, color='darkblue')
plt.axhline(y=0.95, color='r', linestyle='--', label='95% Variance Threshold')
plt.title("PCA Explained Variance: Feature Information Audit")
plt.xlabel("Number of Principal Components")
plt.ylabel("Cumulative Explained Variance")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

# Apply the reduction
X_pca = manager.reduce_dimensions(X_scaled, n_components=0.95, fit=True)

## Step 3: t-SNE Clustering (The Constellation Map)
t-SNE uses non-linear manifold learning to group 'similar' audio clips together. 
* **Inference**: Does Stuttering form its own 'Acoustic Island' or is it mixed into the ocean of Fluent speech?

In [ ]:
# Generating a subset for plot clarity (50/50 balance for visualization)
X_f = X_pca[y == 0]
X_d = X_pca[y == 1]
n_vis = min(1000, len(X_d))
X_vis = np.vstack([X_f[:n_vis], X_d[:n_vis]])
y_vis = np.hstack([np.zeros(n_vis), np.ones(n_vis)])

print("[Plotting]: Generating Non-Linear Projection (t-SNE)...")
tsne = TSNE(n_components=2, perplexity=30, random_state=42)
X_tsne = tsne.fit_transform(X_vis)

plt.figure(figsize=(12, 8))
sns.scatterplot(x=X_tsne[:,0], y=X_tsne[:,1], hue=y_vis, palette="husl", alpha=0.6, s=60)
plt.title("t-SNE Cluster Analysis: Acoustic Distinctness of Stuttering (Red=Stutter, Blue=Fluent)")
plt.xlabel("Visual Component 1")
plt.ylabel("Visual Component 2")
plt.show()

## Step 4: Decision Boundary Visualizations
We map the decision territories of our models. This visually explains why **Non-Linear models** (Curves) outperform **Linear models** (Straight Lines).

In [ ]:
def plot_decision_boundary(model, X_2d, y_2d, title):
    h = .04  # mesh step size
    x_min, x_max = X_2d[:, 0].min() - 1, X_2d[:, 0].max() + 1
    y_min, y_max = X_2d[:, 1].min() - 1, X_2d[:, 1].max() + 1
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
    
    model.train(X_2d, y_2d)
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)
    
    plt.figure(figsize=(10, 8))
    plt.contourf(xx, yy, Z, cmap=plt.cm.RdBu, alpha=0.3)
    sns.scatterplot(x=X_2d[:, 0], y=X_2d[:, 1], hue=y_2d, palette="husl", alpha=0.5, edgecolor="k")
    plt.title(title)
    plt.show()

# Prepare a 2D Projection specifically for boundary mapping (using PCA-2 for logic integrity)
pca_2 = PCA(n_components=2).fit_transform(X_scaled[:2000])
y_2 = y[:2000]

# 1. Logistic Regression Boundary (The baseline straight line)
log_model = LogisticModel("Baseline_Linear")
plot_decision_boundary(log_model, pca_2, y_2, "Logistic Regression Decision Territory (Linear)")

# 2. Kernel RBF SVM Boundary (The complex curve)
svm_model = KernelSVMModel("RBF_Kernel_Complexity", kernel_type='rbf')
plot_decision_boundary(svm_model, pca_2, y_2, "RBF-Kernel SVM Territory (Non-Linear Curve)")